In [23]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [24]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path




In [25]:
try:
    eng = engs.get_engine()
    text_qry = text(engs.load_query('qry_olos.sql'))
    df = pd.read_sql(text_qry, eng)
except Exception as e:
    print (f'Erro ao executar consulta: {e}')

In [26]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [27]:
df.columns

Index(['area', 'base_type', 'campaign_id', 'tablename', 'disposition_nivel_1',
       'data', 'hour', 'tentativas', 'atendidas', 'day_week', 'semana_mes'],
      dtype='object')

In [64]:
df_today = df.copy()
df_today = df_today[df_today['data'] == '2026-05-08']
df_today = df_today.groupby(['tablename']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
df_today['tx'] = ((df_today['atendidas'] /  df_today['tentativas']) * 100 ).round(2)
df_totay = df_today[df_today['tentativas'] > 100]
df_today.sort_values(by='tx',ascending=False)


,tablename,tentativas,atendidas,tx
0,,12,9,75.00
20,_1605_MEDICINA_PAGEVIEW_05052026_V500_pt4,35,3,8.57
1,_1605_ENFERMAGEM_EVENTO_ATUALIZACAOSAUDEMATERN...,658,34,5.17
31,_1605_PSICOLOGIA_EVENTO_SEMINARIOAUTOCUIDADO_0...,22,1,4.55
13,_1605_FISIOTERAPIA_PAGEVIEW_ENGAJADOSMARKETING...,504,20,3.97
3,_1605_ENFERMAGEM_MATERIALRICO_EBOOOK_PROTOCOLO...,27,1,3.70
5,_1605_FISIOTERAPIA_ATIVA_07052026_V500_PT1,27,1,3.70
10,_1605_FISIOTERAPIA_BASELEADS_ENGAJADOSBLACKFRI...,475,17,3.58
11,_1605_FISIOTERAPIA_INATIVO_08052026_V500_pt1,196,7,3.57
19,_1605_MEDICINA_ARTMED_PAGEVIEW_07052026_V500_PT4,198,6,3.03


In [29]:
df_tab = df.copy()
df_tab = df_tab[df_tab['data'] == '2026-05-08']
df_tab = df_tab.groupby(['disposition_nivel_1']).agg(
    quantidade = ('disposition_nivel_1','count')
).reset_index()
df_tab.sort_values(by='quantidade',ascending=False)


,disposition_nivel_1,quantidade
19,NoAnswer,142
6,CarrierMessage,137
1,Announcement,136
28,Silence,120
3,BadNumber,108
2,BadLine,68
29,TransferModule,64
12,IMPRODUTIVO LIGACAO CAIU,60
4,Busy,47
24,RETORNO AGENDADO,38
